# 🤖 Notebook 3 — Model Training & Evaluation
Train and compare Logistic Regression, Random Forest, and XGBoost.

In [4]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

from preprocess import load_data, clean_data, encode_features, split_and_scale

%matplotlib inline
sns.set_theme(style='whitegrid')

In [5]:
# Load & preprocess
df = load_data('/Users/rathanrajdasari/Documents/Customer-Churn-Prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = clean_data(df)
df, _ = encode_features(df)
X_train, X_test, y_train, y_test, scaler = split_and_scale(df)

# SMOTE for class imbalance
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)
print('Class distribution after SMOTE:', dict(zip(*np.unique(y_train, return_counts=True))))

Loaded dataset: 7043 rows, 21 columns
Churn rate: 26.54%
Encoded 15 categorical columns.
Train size: 5634 | Test size: 1409
Class distribution after SMOTE: {0: 4139, 1: 4139}


/Users/rathanrajdasari/Documents/Customer-Churn-Prediction/preprocess.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [6]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
}

# Train and evaluate all
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {'model': model, 'auc': roc_auc_score(y_test, y_proba)}
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred))


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409


--- Random Forest ---
              precision    recall  f1-score   support

           0       0.85      0.84      0.84      1035
           1       0.57      0.59      0.58       374

    accuracy                           0.77      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.77      0.77      0.77      1409


--- XGBoost ---
              precision    recall  f1-score   support

           0       0.85      0.84      0.85      1035
           1       0.57      0.60      0.59       374

    accuracy                           0.78      1409
   macro avg       0.71      0.72      0.72      1409
weigh

/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [08:14:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [9]:
# ROC Curves for all models
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    RocCurveDisplay.from_estimator(res['model'], X_test, y_test, ax=ax, name=name)
ax.set_title('ROC Curve Comparison')
plt.savefig('./reports/roc_curves.png', dpi=150)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/rathanrajdasari/Documents/Customer-Churn-Prediction/reports/roc_curves.png'

In [ ]:
# Confusion matrix for best model (XGBoost)
best_model = results['XGBoost']['model']
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — XGBoost')
plt.savefig('../reports/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# SHAP values — Explainability
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=150)
plt.show()
print('SHAP plot saved.')